# توسعه تمرین کلاسی: چالش شبکه دوبعدی در مسیریاب شهر
**پروژه مسیریاب شهری با در نظر گرفتن موانع و مسیرهای مسدود**

این نوت‌بوک، ارتقای مستقیم تمرین کلاسی عامل هوشمند (جاروبرقی ۴ اتاقه) به **شبکه دوبعدی تقاطع‌های شهری با موانع** است.
در اینجا سه عامل شبیه‌سازی و مقایسه می‌شوند:
1. **عامل تصادفی (Random):** بدون هدف به جهات قانونی تصادفی می‌رود.
2. **عامل واکنشی (Reflex):** حریصانه مسیری را انتخاب می‌کند که فاصله منهتن تا مقصد را کم کند.
3. **عامل مبتنی بر مدل (Model-Based):** حافظه دارد، تقاطع‌های بازدیدشده را ثبت می‌کند و از افتادن در بن‌بست و دور باطل اجتناب می‌کند.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# تنظیمات پایه محیط شهری دوبعدی (مشابه سلول ۱ کلاس)
GRID_SIZE = 6       # ابعاد نقشه شهر (۶ در ۶)
STEPS = 50          # سقف تعداد گام‌ها در هر سفر
RUNS = 100          # تعداد تکرار آزمایش‌ها
START = (0, 0)      # تقاطع شروع
GOAL = (5, 5)       # تقاطع مقصد

def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

In [2]:
# تابع اصلی شبیه‌سازی (ارتقا یافته از simulate کلاس به شبکه دوبعدی)
def simulate(agent_type, seed):
    rng = np.random.default_rng(seed)
    
    # ساخت موانع تصادفی (۲۰٪ احتمال مسدودی معابر)
    obstacles = rng.random((GRID_SIZE, GRID_SIZE)) < 0.20
    obstacles[START] = False
    obstacles[GOAL] = False
    
    pos = START
    visited = set([pos])
    score = 0
    
    actions = {
        "Up":    (-1, 0),
        "Down":  (1, 0),
        "Left":  (0, -1),
        "Right": (0, 1)
    }

    for step in range(STEPS):
        # رسیدن به مقصد
        if pos == GOAL:
            score += (STEPS - step) * 2  # پاداش سرعت
            break
            
        # شناسایی حرکات مجاز (عدم خروج از نقشه و عدم برخورد به مانع)
        valid_moves = {}
        for action_name, (dr, dc) in actions.items():
            nr, nc = pos[0] + dr, pos[1] + dc
            if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
                if not obstacles[nr, nc]:
                    valid_moves[action_name] = (nr, nc)
                    
        if not valid_moves:
            score -= 10  # جریمه بن‌بست
            break

        # انتخاب عمل بر اساس نوع عامل
        if agent_type == "random":
            action = rng.choice(list(valid_moves.keys()))
            
        elif agent_type == "reflex":
            # حریصانه: کاهش فاصله منهتن
            action = min(
                valid_moves.keys(),
                key=lambda a: manhattan_distance(valid_moves[a], GOAL)
            )
            
        elif agent_type == "model_based":
            # مدل‌دار: پرهیز از تکرار خانه‌های بازدیدشده
            unvisited = {a: loc for a, loc in valid_moves.items() if loc not in visited}
            candidates = unvisited if unvisited else valid_moves
            action = min(
                candidates.keys(),
                key=lambda a: manhattan_distance(candidates[a], GOAL)
            )

        pos = valid_moves[action]
        visited.add(pos)
        
        # امتیازدهی بر اساس نزدیک شدن به مقصد
        score += max(0, 10 - manhattan_distance(pos, GOAL))

    return int(score)

In [3]:
# اجرای آزمایشی سریع روی یک دانه تصادفی مشخص
for agent in ["random", "reflex", "model_based"]:
    print(f"Agent: {agent:<12} | Score on seed=42: {simulate(agent, seed=42)}")

In [4]:
# اجرای ۱۰۰ بار آزمایش تکرارپذیر (مشابه سلول ۴ کلاس)
rows = []
for agent in ["random", "reflex", "model_based"]:
    for seed in range(RUNS):
        rows.append({"agent": agent, "score": simulate(agent, seed)})

df = pd.DataFrame(rows)
df.head()

In [5]:
# محاسبه آمار توصیفی میانگین و انحراف معیار (مشابه سلول ۵ کلاس)
summary = df.groupby("agent")["score"].agg(["mean", "std"]).round(1)
print("=== جدول ارزیابی عملکرد عامل‌ها در شبکه ۲ بعدی ===")
summary

In [6]:
# رسم نمودار میله‌ای مقایسه نتایج (مشابه سلول ۶ کلاس)
plt.figure(figsize=(8, 5))
summary["mean"].plot(
    kind="bar", 
    yerr=summary["std"], 
    capsize=5, 
    color=["#3b82f6", "#10b981", "#6366f1"]
)
plt.title("City Router 2D Grid: Average Score over 100 Runs (6x6 Grid, 50 Steps)")
plt.ylabel("Average Performance Score")
plt.xlabel("Agent Architecture")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()